# PRJ-02 · Kaizen server v0: the plan API from a CSV seed

[Open in Colab](https://colab.research.google.com/github/BuiltBySwap/engineering-lab/blob/main/docs/topics/PRJ-02.ipynb)

**Code:** [kaizen/server-ktor](https://github.com/BuiltBySwap/kaizen/tree/main/server-ktor)

**Status card** (edit it; the Progress and Tags pages are built from it):

```yaml
id: PRJ-02
title: "Kaizen server v0: the plan API from a CSV seed"
track: portfolio
date: 2026-10-11
code: "kaizen/server-ktor"
status: partial  # not-started | in-progress | partial | done
score: ""
insight: "Design the API contract first (400 means a malformed request, 404 means a valid request with nothing behind it), load the seed once at startup, compute 'today' in IST, and always check which project you are really editing and running."
tags: [PRJ-02, portfolio, kaizen, ktor, backend, api, csv]
pending:
  - "Emulator check (Chrome at http://10.0.2.2:8080/plan/today) and a screenshot for the README"
  - "Weekly review: English post and the Team Lead role lens (see the day page 2026-10-11)"
  - "MQ118 in my own words"
```

## Goal
Turn the hardcoded `/plan/today` of BE-02 into a real API that serves the 217-day plan, so that the Android app and the React Native companion can both read it.

## The contract (designed before any code)
| Request | Success | Errors |
|---|---|---|
| `GET /health` | 200 `{"status":"ok"}` | |
| `GET /plan/today` | 200 today's plan day (IST) | 404 if today is not in the plan |
| `GET /plan/{date}` (`YYYY-MM-DD`) | 200 the plan day | **400** bad date format, **404** no plan that day |
| `GET /questions/{n}` | 200 the Mobile Exploration question | **400** not a number, **404** no such question |

Every error has one shape, so an app can handle all errors in one place: `{"error": "message"}`.

**400 vs 404.** 400 means the *request is broken* (`/plan/yesterday`: "yesterday" is not a date). 404 means the *request is fine but nothing exists* (`/plan/2030-01-01`). The Kotlin twin: `"abc".toIntOrNull()` returning null is a 400; `map[key]` returning null is a 404.

## Design decisions
- **Load the CSV once, at startup, into memory.** 217 rows, so requests are instant. Trade-off: changing the file needs a restart, until Postgres arrives in PRJ-03.
- **The seed is a classpath resource** (`src/main/resources/seed/plan.csv`), not a file path. A resource is packed into the app, so it works on any machine and in Docker. The code path `"/seed/plan.csv"` is measured from the root of the resources folder.
- **"Today" is computed in `Asia/Kolkata`.** A cloud server runs in UTC, and for 5.5 hours a day the UTC date differs from India's date (the same bug the `Clock` test caught in PRJ-01).
- **Read CSV columns by name, not by position**, so reordering or adding columns does not break the code.
- **Route matching:** when `/plan/today` and `/plan/{date}` both fit, Ktor runs the more specific constant route.
- **Paths are measured from `src/main/resources`**, and the program runs from the build output (`build/resources/main`), which can be stale: `./gradlew clean processResources`.

## Why a plain `split(",")` cannot read this CSV
Fields are wrapped in quotes; a comma inside quotes is data; a line break inside quotes is data (the `steps` field has 7 lines); a quote inside a field is written twice (`""`). So the parser is a small state machine with one flag, `inQuotes`. Trace on `a,"b,c",d`:

| char | in quotes? | action |
|---|---|---|
| `a` | no | add to the field |
| `,` | no | end the field (`a`) |
| `"` | no → yes | flip the flag |
| `b` `,` `c` | yes | add all to the field (the comma is data) |
| `"` | yes → no | flip the flag |
| `,` | no | end the field (`b,c`) |
| `d` | no | add to the field |

Run it (Kotlin cell, in Colab or locally; run the setup cell first):

In [ ]:
# Setup: run once per session. Adds the %%kotlin cell magic. Works in Colab and in VS Code / Jupyter.
import os, urllib.request
SRC = next((p for p in ("tools/kotlin_magic.py", "../../tools/kotlin_magic.py") if os.path.exists(p)), None)
if SRC is None:                                   # e.g. Colab: download it from GitHub
    SRC = "kotlin_magic.py"
    urllib.request.urlretrieve("https://raw.githubusercontent.com/BuiltBySwap/engineering-lab/main/tools/kotlin_magic.py", SRC)
%run $SRC

In [ ]:
%%kotlin
import kotlin.test.assertEquals
import kotlin.test.assertNull

fun parseCsv(text: String): List<List<String>> {
    val rows = mutableListOf<List<String>>()
    var row = mutableListOf<String>()
    val field = StringBuilder()
    var inQuotes = false
    var i = 0
    while (i < text.length) {
        val c = text[i]
        when {
            inQuotes && c == '"' && i + 1 < text.length && text[i + 1] == '"' -> { field.append('"'); i++ }
            c == '"' -> inQuotes = !inQuotes
            !inQuotes && c == ',' -> { row.add(field.toString()); field.clear() }
            !inQuotes && (c == '\n' || c == '\r') -> {
                if (c == '\r' && i + 1 < text.length && text[i + 1] == '\n') i++
                row.add(field.toString()); field.clear()
                rows.add(row); row = mutableListOf()
            }
            else -> field.append(c)
        }
        i++
    }
    if (field.isNotEmpty() || row.isNotEmpty()) { row.add(field.toString()); rows.add(row) }
    return rows
}

// 1. the tricky cases from the real file
assertEquals(listOf(listOf("a", "b", "c")), parseCsv("a,b,c"))
assertEquals(listOf(listOf("a", "b,c", "d")), parseCsv("a,\"b,c\",d\n"))
assertEquals(listOf(listOf("say \"hi\"")), parseCsv("\"say \"\"hi\"\"\""))
assertEquals(listOf(listOf("line1\nline2", "x"), listOf("1", "2")), parseCsv("\"line1\nline2\",x\r\n1,2\r\n"))
println("parser: 4 checks passed")

// 2. a tiny repository over an inline CSV (a trimmed copy of the real PlanRepository)
val csv = "\"date\",\"id\",\"topic\",\"mobile_q\"\r\n" +
    "\"2026-10-11\",\"PRJ-02\",\"Kaizen server\",\"MQ118 [Notifications] Why do notifications sometimes arrive late?\nMQ139 [Notifications] How does FCM know which device to send to?\"\r\n" +
    "\"2026-10-12\",\"DSA-02\",\"Binary search on the answer\",\"\"\r\n"

val rows = parseCsv(csv)
val header = rows.first()
val index = header.withIndex().associate { (i, name) -> name to i }      // read columns by NAME
fun List<String>.col(name: String) = this[index.getValue(name)]

val byDate = rows.drop(1).associateBy { it.col("date") }
val pattern = Regex("""MQ(\d+) \[(.*?)] (.*)""")
val questions = rows.drop(1).flatMap { r ->
    r.col("mobile_q").lines().mapNotNull { line ->
        pattern.matchEntire(line.trim())?.let { m -> m.groupValues[1].toInt() to m.groupValues[3] }
    }
}.toMap()

assertEquals("PRJ-02", byDate["2026-10-11"]?.col("id"))
assertNull(byDate["2030-01-01"])                                     // the 404 case
assertEquals("Why do notifications sometimes arrive late?", questions[118])
assertNull(questions[99999])                                          // the 404 case
println("repository: lookups work, ${questions.size} questions indexed")

## Why "today" needs a time zone
For 5.5 hours of every day (18:30 to 24:00 UTC) the UTC date is one day behind India's date. A server in the cloud runs in UTC, so `LocalDate.now()` would give the wrong day. The code asks for `LocalDate.now(ZoneId.of("Asia/Kolkata"))` explicitly.

In [ ]:
from datetime import datetime, timezone, timedelta

utc = datetime(2026, 10, 11, 20, 0, tzinfo=timezone.utc)                 # 20:00 UTC on 11 Oct
ist = utc.astimezone(timezone(timedelta(hours=5, minutes=30)))           # the same moment in India
print("UTC date:", utc.date(), "   IST date:", ist.date())
assert str(utc.date()) == "2026-10-11" and str(ist.date()) == "2026-10-12"

## The code, file by file (the real project: `kaizen/server-ktor`)

**`Models.kt`** (added next to `Health`):
```kotlin
@Serializable
data class PlanDay(
    val date: String, val id: String, val track: String, val priority: String, val load: String,
    val hours: Double, val topic: String, val why: String, val steps: List<String>,
    val tool: String, val leetcode: String, val mobileQuestion: String, val english: String,
)
@Serializable data class Question(val number: Int, val category: String, val text: String, val date: String)
@Serializable data class ApiError(val error: String)
```
Dates travel as ISO-8601 text (`"2026-10-11"`), because JSON has no date type. The property names are the JSON keys, so they are now a **contract**: renaming one breaks installed apps.

**`Csv.kt`**: the `parseCsv` function above. **`PlanRepository.kt`**: `parseCsv` once at startup; `header.withIndex().associate` for the column index; `associateBy { it.date }` for an instant lookup by date; a regex to pull `MQ<n> [category] text` out of each day; `forDate` and `question` return `null` when nothing matches, which is the signal for 404; a `fromResource(path)` factory that reads the CSV from the classpath and fails loudly if it is missing.

**`Routing.kt`**:
```kotlin
fun Application.configureRouting() {
    val repository = PlanRepository.fromResource("/seed/plan.csv")   // read ONCE, at startup
    routing {
        get("/health") { call.respond(Health("ok")) }
        get("/plan/today") {
            val today = LocalDate.now(IST).toString()
            val day = repository.forDate(today)
            if (day == null) call.respond(HttpStatusCode.NotFound, ApiError("no plan for $today")) else call.respond(day)
        }
        get("/plan/{date}") {
            val date = runCatching { LocalDate.parse(call.parameters["date"].orEmpty()) }.getOrNull()
            if (date == null) { call.respond(HttpStatusCode.BadRequest, ApiError("date must look like 2026-10-11")); return@get }
            val day = repository.forDate(date.toString())
            if (day == null) call.respond(HttpStatusCode.NotFound, ApiError("no plan for $date")) else call.respond(day)
        }
        get("/questions/{n}") {
            val number = call.parameters["n"]?.toIntOrNull()
            if (number == null) { call.respond(HttpStatusCode.BadRequest, ApiError("n must be a number")); return@get }
            val question = repository.question(number)
            if (question == null) call.respond(HttpStatusCode.NotFound, ApiError("no question $number")) else call.respond(question)
        }
    }
}
```
`return@get` leaves the handler early, because a request can only be answered once.

**Try it:**
```bash
cd ~/dev/kaizen/server-ktor && ./gradlew run
curl -i http://localhost:8080/plan/2026-10-11      # 200 PRJ-02
curl -i http://localhost:8080/plan/yesterday       # 400 {"error": "date must look like 2026-10-11"}
curl -i http://localhost:8080/plan/2030-01-01      # 404 {"error": "no plan for 2030-01-01"}
curl -i http://localhost:8080/questions/433        # 200 the On-device AI question
```
In the Android emulator use `http://10.0.2.2:8080`.

## Bug diary
| Error or symptom | Cause | Fix |
|---|---|---|
| `IllegalStateException: Resource not found: /seed/plan.csv` | The file was in `src/main/resources/seed/`, but the build output (`build/resources/main`) was stale and never got it | `./gradlew clean processResources`, then check `ls build/resources/main/seed` |
| `BindException: Address already in use` | An older server still held port 8080 | `kill $(lsof -nP -t -iTCP:8080 -sTCP:LISTEN)`; start the server from one place only |
| `kill <PID>`: `zsh: parse error` | I typed the placeholder literally. Angle brackets mean "put your value here" | Use `kill $(lsof -nP -t -iTCP:8080 -sTCP:LISTEN)` |
| 404 with an empty body, and `/plan/today` still returned the old BE-02 row | **I was editing and running the old copy** in `~/Downloads/kaizen-server`, not `~/dev/kaizen/server-ktor` | See below |

**How the wrong-project bug was found:** the running server logged my requests, so it was alive and on the right port. The code I pasted had `configureRouting` at line 12 while `grep` on the real project said line 7, and `find` showed no `Csv.kt` or `PlanRepository.kt` there. Evidence from the disk beat guessing. Fix: copy the files into the right project, rename the old folder (`kaizen-server-OLD`), and read the project name in the IntelliJ window title before editing.

**Empty 404 vs a JSON 404:** Ktor's default for "no route matched" is a 404 with no body. My own 404s always carry `{"error": ...}`. The body tells you which code produced the response.

## Quiz and mistakes
- Said 404 for `GET /plan/yesterday`: it should be **400**, because "yesterday" is not a date, so the request itself is malformed.
- Right: `/plan/2030-01-01` is 404; a CSV needs a parser because quotes contain commas and line breaks; columns are read by name so reordering does not break the code; `/plan/today` wins over `/plan/{date}` because the constant route is more specific; `return@get` stops the handler so it cannot respond twice.

## In my own words
✏️ Why does an API contract matter more for a mobile app than for a website?

## Next
**PRJ-03** (18 Oct): replace the CSV with Postgres using Exposed, Flyway migrations and docker-compose.